# GraviGraph - Graph analytics and graph neural networks

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/GraviFrame/bin/Release/net10.0/Gravicode.Science.GraviFrame.dll"
#r "../src/GraviLearn/bin/Release/net10.0/Gravicode.Science.GraviLearn.dll"
#r "../src/GraviText/bin/Release/net10.0/Gravicode.Science.GraviText.dll"
#r "../src/GraviGraph/bin/Release/net10.0/Gravicode.Science.GraviGraph.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviGraph;
using Gravicode.Science.GraviGraph.Algorithms;
using Gravicode.Science.GraviGraph.Embeddings;
using Gravicode.Science.GraviGraph.Neural;
using Gravicode.Science.GraviNum;

var cora = Graph.Load("../datasets/cora_graph.json");
Console.WriteLine(cora);
Console.WriteLine($"features: {cora.NodeFeatures!.Shape[0]} x {cora.NodeFeatures.Shape[1]}");
Console.WriteLine($"classes : {string.Join(", ", cora.Classes)}");

## PageRank

In [ ]:
var rank = GraphAlgorithms.PageRank(cora);
Console.WriteLine($"total mass {rank.Sum():F6}");

foreach (var node in Enumerable.Range(0, cora.NodeCount).OrderByDescending(i => rank.At(i)).Take(10))
    Console.WriteLine($"node {node,6}  {rank.At(node):F6}  degree {cora.Degree(node),4}  {cora.Classes[cora.NodeLabels[node]]}");

## Structure

On a directed citation graph, connectivity means *weak* connectivity - edges followed both ways.

In [ ]:
var (weak, component) = GraphAlgorithms.ConnectedComponents(cora);
Console.WriteLine($"weakly connected  : {weak} components, largest {component.GroupBy(c => c).Max(g => g.Count())} nodes");
Console.WriteLine($"strongly connected: {GraphAlgorithms.StronglyConnectedComponents(cora).Count} components");
Console.WriteLine($"average clustering: {GraphAlgorithms.AverageClusteringCoefficient(cora):F4}");
Console.WriteLine($"modularity of the true labels: {GraphAlgorithms.Modularity(cora, cora.NodeLabels):F4}");

## Training a GCN

Only 140 of 2708 papers are labelled - the semi-supervised setting the architecture was designed for.

In [ ]:
var rng = new GraviRandom(42);
var order = rng.Permutation(cora.NodeCount);
var train = order.Take(140).ToArray();
var test = order.Skip(1708).ToArray();

var gcn = new GraphConvolutionalNetwork(hiddenSize: 16, learningRate: 0.05, epochs: 60, seed: 42)
    .Train(cora, train, features: cora.NodeFeatures);

Console.WriteLine($"train accuracy: {gcn.Score(cora, train):P1}");
Console.WriteLine($"test accuracy : {gcn.Score(cora, test):P1}");
Console.WriteLine($"majority-class baseline: {cora.NodeLabels.GroupBy(l => l).Max(g => g.Count()) / (double)cora.NodeCount:P1}");

In [ ]:
var loss = gcn.History!.Loss.ToArray();
var epochs = Enumerable.Range(0, loss.Length).Select(i => (double)i).ToArray();

var plot = new ScottPlot.Plot();
var line = plot.Add.Scatter(epochs, loss);
line.MarkerSize = 0;
plot.Title("GCN training loss");
plot.XLabel("epoch"); plot.YLabel("cross-entropy");
plot.GetImageHtml(800, 450)

## Node embeddings

Random walks need the undirected view, otherwise most walks stop after a step.

In [ ]:
var (_, comp) = GraphAlgorithms.ConnectedComponents(cora);
var biggest = comp.Select((c, i) => (c, i)).GroupBy(t => t.c).MaxBy(g => g.Count())!.Select(t => t.i).Take(400).ToArray();
var sub = cora.Subgraph(biggest).AsUndirected();

var embeddings = new Node2Vec(dimensions: 64, p: 1.0, q: 0.5, walksPerNode: 6, walkLength: 20, epochs: 3, seed: 42).Train(sub);

double same = 0, other = 0; int sameN = 0, otherN = 0;
for (var i = 0; i < sub.NodeCount; i++)
    for (var j = i + 1; j < sub.NodeCount; j++)
    {
        var s = embeddings.Similarity(i, j);
        if (sub.NodeLabels[i] == sub.NodeLabels[j]) { same += s; sameN++; } else { other += s; otherN++; }
    }
Console.WriteLine($"same-topic cosine advantage: {same / sameN - other / otherN:F4} (positive means the embedding found the topics)");